In [1]:
"""
HemMaskNet - Complete Implementation for Reviewer 2 (BUGFIXED)
==============================================================
Fixes:
1. BatchNorm crash on last batch (drop_last=True)
2. Auto-detects annotation format: drop-level reactions vs image-level labels
3. Better label derivation with validation warnings

Directory Structure (auto-detected, case-insensitive):
/home/fawadsalamkhan/MyProjects/BloodGroup/
├── train/images/  train/labels/
├── valid/images/  valid/labels/
└── test/images/   test/labels/
"""

import os
import time
import random
import warnings
import json
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Tuple, List, Dict, Optional

import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
import scipy.stats as stats

from PIL import Image, ImageEnhance, ImageFilter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import psutil
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False

warnings.filterwarnings('ignore')

# =============================================================================
# HELPERS
# =============================================================================

def find_subdir(parent: str, target_names: List[str]) -> Optional[str]:
    if not os.path.exists(parent):
        return None
    for entry in os.listdir(parent):
        entry_path = os.path.join(parent, entry)
        if os.path.isdir(entry_path) and entry.lower() in [t.lower() for t in target_names]:
            return entry_path
    return None

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class Config:
    base_dir: str = '/home/fawadsalamkhan/MyProjects/BloodGroup'
    output_dir: str = '/home/fawadsalamkhan/MyProjects/BloodGroup/results_reviewer2'

    img_size: int = 224
    num_classes: int = 8
    batch_size: int = 8
    num_workers: int = 0  # safer for debugging

    epochs: int = 30
    lr: float = 5.699e-4
    weight_decay: float = 8.545e-4
    dropout1: float = 0.50
    dropout2: float = 0.319
    hidden1: int = 512
    hidden2: int = 64
    optimizer_name: str = 'AdamW'
    scheduler_patience: int = 7

    n_splits: int = 5
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Mask encoding
    anti_a_intensity: int = 200
    anti_b_intensity: int = 150
    anti_d_intensity: int = 255

    # Annotation format:
    # 'drop_reactions' = paper format: class IDs 0-5 per drop (Anti-A pos/neg, etc.)
    # 'image_level'    = one class ID per image = blood group directly (0-7)
    # 'auto'           = try to auto-detect
    annotation_format: str = 'auto'

    test_mask_error_rates: List[float] = field(default_factory=lambda: [0.0, 0.1, 0.2, 0.3])
    test_lighting_variations: List[str] = field(default_factory=lambda: ['normal', 'bright', 'dim', 'contrast', 'blur'])

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)

# =============================================================================
# SEMANTIC MASK GENERATOR
# =============================================================================

class SemanticMaskGenerator:
    def __init__(self, config: Config):
        self.config = config
        # For drop-level reaction format (paper)
        self.intensity_map = {
            0: config.anti_a_intensity,
            1: config.anti_a_intensity,
            2: config.anti_b_intensity,
            3: config.anti_b_intensity,
            4: config.anti_d_intensity,
            5: config.anti_d_intensity,
        }
        # For image-level format (Roboflow-style): drops are just regions
        self.drop_intensity_map = {
            0: config.anti_a_intensity,
            1: config.anti_b_intensity,
            2: config.anti_d_intensity,
        }

    def generate_mask(self, image_shape: Tuple[int, int],
                      yolo_annotations: List[List[float]],
                      error_rate: float = 0.0,
                      format_type: str = 'drop_reactions') -> np.ndarray:
        h, w = image_shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)

        for ann in yolo_annotations:
            if len(ann) < 5:
                continue
            class_id = int(ann[0])
            x_c, y_c, bw, bh = ann[1:5]
            x_c *= w
            y_c *= h
            bw *= w
            bh *= h
            center = (int(x_c), int(y_c))
            axes = (int(bw / 2), int(bh / 2))

            if format_type == 'image_level':
                # Drops are just spatial regions, use order of appearance
                # or map by class_id if available
                intensity = self.drop_intensity_map.get(
                    min(class_id, 2), 128
                )
            else:
                intensity = self.intensity_map.get(class_id, 128)

            cv2.ellipse(mask, center, axes, 0, 0, 360, intensity, -1)

        if error_rate > 0 and np.random.rand() < error_rate:
            mask = self._corrupt_mask(mask)
        return mask

    def _corrupt_mask(self, mask: np.ndarray) -> np.ndarray:
        h, w = mask.shape
        drop_regions = [200, 150, 255]
        if len(drop_regions) > 0:
            target = np.random.choice(drop_regions)
            mask[mask == target] = 0
        cx, cy = np.random.randint(0, w), np.random.randint(0, h)
        cv2.circle(mask, (cx, cy), np.random.randint(10, 30),
                   np.random.choice(drop_regions), -1)
        return mask

# =============================================================================
# DATASET with auto format detection
# =============================================================================

class BloodGroupDataset(Dataset):
    def __init__(self, split_dir: str, config: Config, transform=None,
                 mode='train', lighting_variant='normal', mask_error_rate=0.0):
        self.split_dir = split_dir
        self.config = config
        self.transform = transform
        self.mode = mode
        self.lighting_variant = lighting_variant
        self.mask_error_rate = mask_error_rate
        self.mask_gen = SemanticMaskGenerator(config)
        self.class_names = ['A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-']

        self.image_dir = find_subdir(split_dir, ['Images', 'images', 'IMG', 'img', 'Image'])
        self.label_dir = find_subdir(split_dir, ['labels', 'Labels', 'label', 'Label'])

        if self.image_dir is None:
            print(f"WARNING: No image subdirectory found in {split_dir}")
            print(f"  Contents: {os.listdir(split_dir) if os.path.exists(split_dir) else 'DIR NOT FOUND'}")
        if self.label_dir is None:
            print(f"WARNING: No label subdirectory found in {split_dir}")
            print(f"  Contents: {os.listdir(split_dir) if os.path.exists(split_dir) else 'DIR NOT FOUND'}")

        self.samples = self._load_samples()
        print(f"[{mode}] Loaded {len(self.samples)} samples from {split_dir}")
        if len(self.samples) > 0:
            print(f"  Image dir: {self.image_dir}")
            print(f"  Label dir: {self.label_dir}")
            dist = self.get_class_distribution()
            print(f"  Class distribution: {dist}")
            # Warning if all same class
            if len(dist) == 1:
                print(f"  WARNING: All samples have the SAME label! Check annotation format.")
                print(f"  Try setting config.annotation_format='image_level' if using Roboflow-style labels.")

    def _load_samples(self) -> List[Dict]:
        samples = []
        if self.image_dir is None or self.label_dir is None:
            return samples

        image_files = sorted([f for f in os.listdir(self.image_dir)
                              if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'))])

        print(f"  Found {len(image_files)} image files in {self.image_dir}")

        # First pass: detect annotation format
        format_type = self.config.annotation_format
        if format_type == 'auto':
            format_type = self._detect_format(image_files[:20])
            print(f"  Auto-detected annotation format: {format_type}")
        self.format_type = format_type

        for img_name in image_files:
            base = os.path.splitext(img_name)[0]
            ann_path = os.path.join(self.label_dir, base + '.txt')
            img_path = os.path.join(self.image_dir, img_name)

            if not os.path.exists(ann_path):
                found = False
                for ext in ['.txt', '.TXT']:
                    alt = os.path.join(self.label_dir, base + ext)
                    if os.path.exists(alt):
                        ann_path = alt
                        found = True
                        break
                if not found:
                    continue

            annotations = []
            try:
                with open(ann_path, 'r') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = list(map(float, line.split()))
                        if len(parts) >= 5:
                            annotations.append(parts)
            except Exception as e:
                print(f"Error reading {ann_path}: {e}")
                continue

            if len(annotations) == 0:
                continue

            # Derive label based on format
            if self.format_type == 'image_level':
                # Single annotation per image = blood group class
                label = int(annotations[0][0])
                if label >= self.config.num_classes:
                    continue
            else:
                # Drop-level reactions (paper format)
                if len(annotations) < 3:
                    continue
                label = self._derive_label_reactions(annotations)
                if label is None:
                    continue

            samples.append({
                'image': img_path,
                'annotations': annotations,
                'label': label,
                'name': base
            })
        return samples

    def _detect_format(self, sample_images: List[str]) -> str:
        """Auto-detect if annotations are drop-reactions or image-level."""
        reaction_class_ids = set()
        all_class_ids = set()
        n_drops_per_file = []

        for img_name in sample_images:
            base = os.path.splitext(img_name)[0]
            ann_path = os.path.join(self.label_dir, base + '.txt')
            if not os.path.exists(ann_path):
                continue
            try:
                with open(ann_path, 'r') as f:
                    lines = [l.strip() for l in f if l.strip()]
                    n_drops_per_file.append(len(lines))
                    for line in lines:
                        parts = list(map(float, line.split()))
                        if len(parts) >= 5:
                            cid = int(parts[0])
                            all_class_ids.add(cid)
                            if len(lines) >= 3:  # likely drop-level
                                reaction_class_ids.add(cid)
            except:
                continue

        if len(n_drops_per_file) == 0:
            return 'drop_reactions'

        avg_drops = np.mean(n_drops_per_file)
        max_class = max(all_class_ids) if all_class_ids else 0

        # Heuristic: if most files have 1 annotation and class IDs go up to 7, it's image-level
        if avg_drops < 1.5 and max_class >= 5:
            return 'image_level'
        # If class IDs are mostly 0-5 and 3 drops per file, it's drop-reactions
        if avg_drops >= 2.5 and max_class <= 5:
            return 'drop_reactions'
        # Default to image_level if many classes detected
        if max_class >= 7:
            return 'image_level'
        return 'drop_reactions'

    def _derive_label_reactions(self, annotations: List[List[float]]) -> Optional[int]:
        a_agglut = False
        b_agglut = False
        d_agglut = False

        for ann in annotations:
            class_id = int(ann[0])
            if class_id == 0:   # Anti-A Positive
                a_agglut = True
            elif class_id == 2: # Anti-B Positive
                b_agglut = True
            elif class_id == 4: # Anti-D Positive
                d_agglut = True

        if a_agglut and b_agglut and d_agglut:
            return 6   # AB+
        elif a_agglut and b_agglut and not d_agglut:
            return 7   # AB-
        elif a_agglut and not b_agglut and d_agglut:
            return 0   # A+
        elif a_agglut and not b_agglut and not d_agglut:
            return 1   # A-
        elif not a_agglut and b_agglut and d_agglut:
            return 2   # B+
        elif not a_agglut and b_agglut and not d_agglut:
            return 3   # B-
        elif not a_agglut and not b_agglut and d_agglut:
            return 4   # O+
        elif not a_agglut and not b_agglut and not d_agglut:
            return 5   # O-
        return None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample['image']).convert('RGB')
        image_np = np.array(image)
        image = self._apply_lighting(image, self.lighting_variant)

        mask = self.mask_gen.generate_mask(
            image_np.shape, sample['annotations'],
            error_rate=self.mask_error_rate,
            format_type=self.format_type
        )
        mask = Image.fromarray(mask)

        image = image.resize((self.config.img_size, self.config.img_size))
        mask = mask.resize((self.config.img_size, self.config.img_size), Image.NEAREST)

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        mask = transforms.ToTensor()(mask)
        label = sample['label']
        return image, mask, label

    def _apply_lighting(self, image: Image.Image, variant: str) -> Image.Image:
        if variant == 'normal':
            return image
        elif variant == 'bright':
            return ImageEnhance.Brightness(image).enhance(1.5)
        elif variant == 'dim':
            return ImageEnhance.Brightness(image).enhance(0.6)
        elif variant == 'contrast':
            return ImageEnhance.Contrast(image).enhance(1.8)
        elif variant == 'blur':
            return image.filter(ImageFilter.GaussianBlur(radius=2))
        return image

    def get_class_distribution(self) -> Dict[int, int]:
        dist = defaultdict(int)
        for s in self.samples:
            dist[s['label']] += 1
        return dict(dist)

# =============================================================================
# MODELS
# =============================================================================

class MaskEncoder(nn.Module):
    def __init__(self, out_dim: int = 128):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

class HemMaskNet(nn.Module):
    def __init__(self, config: Config, num_classes: int = 8):
        super().__init__()
        self.image_encoder = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.image_encoder.classifier = nn.Identity()
        self.image_dim = 1280

        self.mask_encoder = MaskEncoder(out_dim=128)
        self.mask_dim = 128
        self.fused_dim = self.image_dim + self.mask_dim

        self.classifier = nn.Sequential(
            nn.Linear(self.fused_dim, config.hidden1),
            nn.BatchNorm1d(config.hidden1), nn.ReLU(), nn.Dropout(config.dropout1),
            nn.Linear(config.hidden1, config.hidden2),
            nn.BatchNorm1d(config.hidden2), nn.ReLU(), nn.Dropout(config.dropout2),
            nn.Linear(config.hidden2, num_classes)
        )

    def forward(self, image, mask):
        img_feat = self.image_encoder(image)
        mask_feat = self.mask_encoder(mask)
        fused = torch.cat([img_feat, mask_feat], dim=1)
        return self.classifier(fused)

class EfficientNetOnly(nn.Module):
    def __init__(self, config: Config, num_classes: int = 8):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(1280, config.hidden1),
            nn.BatchNorm1d(config.hidden1), nn.ReLU(), nn.Dropout(config.dropout1),
            nn.Linear(config.hidden1, config.hidden2),
            nn.BatchNorm1d(config.hidden2), nn.ReLU(), nn.Dropout(config.dropout2),
            nn.Linear(config.hidden2, num_classes)
        )

    def forward(self, image, mask=None):
        return self.classifier(self.backbone(image))

class ResNet50Baseline(nn.Module):
    def __init__(self, config: Config, num_classes: int = 8):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V1')
        self.backbone.fc = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(2048, config.hidden1),
            nn.BatchNorm1d(config.hidden1), nn.ReLU(), nn.Dropout(config.dropout1),
            nn.Linear(config.hidden1, num_classes)
        )

    def forward(self, image, mask=None):
        return self.classifier(self.backbone(image))

class MobileNetBaseline(nn.Module):
    def __init__(self, config: Config, num_classes: int = 8):
        super().__init__()
        self.backbone = models.mobilenet_v2(weights='IMAGENET1K_V1')
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, mask=None):
        return self.classifier(self.backbone(image))

class ViTB16Baseline(nn.Module):
    def __init__(self, config: Config, num_classes: int = 8):
        super().__init__()
        self.backbone = models.vit_b_16(weights='IMAGENET1K_V1')
        self.backbone.heads = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Linear(768, config.hidden1), nn.ReLU(), nn.Dropout(config.dropout1),
            nn.Linear(config.hidden1, num_classes)
        )

    def forward(self, image, mask=None):
        return self.classifier(self.backbone(image))

# =============================================================================
# CLASSICAL BASELINE
# =============================================================================

class ClassicalBaseline:
    def __init__(self, img_size=224):
        self.img_size = img_size
        self.clf = make_pipeline(StandardScaler(), PCA(n_components=100),
                                 SVC(kernel='rbf', probability=True, C=10))

    def _extract(self, img_path: str) -> np.ndarray:
        img = cv2.imread(img_path)
        if img is None:
            return np.zeros(200, dtype=np.float32)
        img = cv2.resize(img, (self.img_size, self.img_size))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        features = []
        for i in range(3):
            hist = cv2.calcHist([img], [i], None, [32], [0, 256]).flatten()
            features.extend(hist)
        edges = cv2.Canny(gray, 50, 150)
        features.append(np.sum(edges > 0) / (self.img_size ** 2))
        features.append(cv2.Laplacian(gray, cv2.CV_64F).var())
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        features.append(len(contours))
        if contours:
            areas = [cv2.contourArea(c) for c in contours]
            features.extend([np.mean(areas), np.std(areas), np.max(areas)])
        else:
            features.extend([0, 0, 0])
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        for i in range(3):
            features.extend([np.mean(hsv[:,:,i]), np.std(hsv[:,:,i])])
        return np.array(features, dtype=np.float32)

    def fit(self, dataset: BloodGroupDataset):
        X, y = [], []
        for s in dataset.samples:
            X.append(self._extract(s['image']))
            y.append(s['label'])
        self.clf.fit(np.array(X), np.array(y))
        return self

    def predict(self, dataset: BloodGroupDataset):
        X, y_true = [], []
        for s in dataset.samples:
            X.append(self._extract(s['image']))
            y_true.append(s['label'])
        y_pred = self.clf.predict(np.array(X))
        return np.array(y_true), y_pred

# =============================================================================
# TRAINER
# =============================================================================

class Trainer:
    def __init__(self, config: Config, model: nn.Module, model_name: str):
        self.config = config
        self.model = model.to(config.device)
        self.model_name = model_name
        self.criterion = nn.CrossEntropyLoss()

        if config.optimizer_name == 'AdamW':
            self.optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr,
                                                 weight_decay=config.weight_decay)
        elif config.optimizer_name == 'Adam':
            self.optimizer = torch.optim.Adam(model.parameters(), lr=config.lr,
                                              weight_decay=config.weight_decay)
        else:
            self.optimizer = torch.optim.SGD(model.parameters(), lr=config.lr,
                                             momentum=0.9, weight_decay=config.weight_decay)

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='min', patience=config.scheduler_patience, factor=0.5
        )
        self.best_val_acc = 0.0
        self.history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    def train_epoch(self, loader: DataLoader):
        self.model.train()
        total_loss, correct, total = 0, 0, 0
        for images, masks, labels in loader:
            images = images.to(self.config.device)
            masks = masks.to(self.config.device)
            labels = labels.to(self.config.device)

            self.optimizer.zero_grad()
            outputs = self.model(images, masks)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
        return total_loss / total, correct / total

    def evaluate(self, loader: DataLoader):
        self.model.eval()
        total_loss, correct, total = 0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, masks, labels in loader:
                images = images.to(self.config.device)
                masks = masks.to(self.config.device)
                labels = labels.to(self.config.device)
                outputs = self.model(images, masks)
                loss = self.criterion(outputs, labels)

                total_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        return total_loss / total, correct / total, np.array(all_preds), np.array(all_labels)

    def fit(self, train_loader: DataLoader, val_loader: DataLoader):
        for epoch in range(self.config.epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc, _, _ = self.evaluate(val_loader)
            self.scheduler.step(val_loss)

            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)

            if val_acc > self.best_val_acc:
                self.best_val_acc = val_acc

            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{self.model_name}] Epoch {epoch+1}/{self.config.epochs} | "
                      f"Train: {train_loss:.4f}/{train_acc:.4f} | "
                      f"Val: {val_loss:.4f}/{val_acc:.4f}")
        return self.history

# =============================================================================
# CROSS-VALIDATION
# =============================================================================

class CrossValidator:
    def __init__(self, config: Config):
        self.config = config
        self.results = {}

    def _wilson_ci(self, successes, n, confidence=0.95):
        if n == 0:
            return (0.0, 1.0)
        z = stats.norm.ppf(1 - (1 - confidence) / 2)
        p = successes / n
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        hw = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
        return (max(0, centre - hw), min(1, centre + hw))

    def run_cv(self, dataset: BloodGroupDataset, model_class, model_name: str) -> Dict:
        print(f"\n{'='*70}")
        print(f"Cross-Validation: {model_name}")
        print(f"{'='*70}")

        labels = [s['label'] for s in dataset.samples]
        skf = StratifiedKFold(n_splits=self.config.n_splits, shuffle=True, random_state=42)

        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        fold_accs = []
        fold_f1s = []
        all_y_true = []
        all_y_pred = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
            print(f"\nFold {fold + 1}/{self.config.n_splits}")

            train_samples = [dataset.samples[i] for i in train_idx]
            val_samples = [dataset.samples[i] for i in val_idx]

            train_ds = BloodGroupDataset(dataset.split_dir, self.config,
                                          transform=transform, mode='train')
            train_ds.samples = train_samples
            train_ds.format_type = dataset.format_type

            val_ds = BloodGroupDataset(dataset.split_dir, self.config,
                                        transform=transform, mode='val')
            val_ds.samples = val_samples
            val_ds.format_type = dataset.format_type

            train_labels = [s['label'] for s in train_samples]
            class_counts = np.bincount(train_labels, minlength=self.config.num_classes)
            weights = 1.0 / (class_counts + 1e-6)
            sample_weights = [weights[l] for l in train_labels]
            sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

            # FIX: drop_last=True prevents BatchNorm crash on batch size 1
            train_loader = DataLoader(train_ds, batch_size=self.config.batch_size,
                                      sampler=sampler, num_workers=0, drop_last=True)
            val_loader = DataLoader(val_ds, batch_size=self.config.batch_size,
                                    shuffle=False, num_workers=0)

            model = model_class(self.config, num_classes=self.config.num_classes)
            trainer = Trainer(self.config, model, f"{model_name}_fold{fold}")
            trainer.fit(train_loader, val_loader)

            _, val_acc, preds, labels_arr = trainer.evaluate(val_loader)
            fold_accs.append(val_acc)
            _, _, f1, _ = precision_recall_fscore_support(labels_arr, preds, average='macro', zero_division=0)
            fold_f1s.append(f1)

            all_y_true.extend(labels_arr)
            all_y_pred.extend(preds)
            print(f"  Fold {fold+1} -> Acc: {val_acc:.4f}, F1: {f1:.4f}")

        acc_mean = np.mean(fold_accs)
        acc_std = np.std(fold_accs)
        f1_mean = np.mean(fold_f1s)
        f1_std = np.std(fold_f1s)
        n = len(fold_accs)
        t_val = stats.t.ppf(0.975, n - 1)
        acc_ci = t_val * acc_std / np.sqrt(n)
        f1_ci = t_val * f1_std / np.sqrt(n)

        total_correct = np.sum(np.array(all_y_true) == np.array(all_y_pred))
        wilson = self._wilson_ci(total_correct, len(all_y_true))

        results = {
            'model_name': model_name,
            'accuracy_mean': acc_mean,
            'accuracy_std': acc_std,
            'accuracy_95ci': (acc_mean - acc_ci, acc_mean + acc_ci),
            'f1_mean': f1_mean,
            'f1_std': f1_std,
            'f1_95ci': (f1_mean - f1_ci, f1_mean + f1_ci),
            'wilson_95ci': wilson,
            'fold_accuracies': fold_accs,
            'fold_f1s': fold_f1s,
            'confusion_matrix': confusion_matrix(all_y_true, all_y_pred).tolist(),
            'per_class_report': classification_report(
                all_y_true, all_y_pred,
                target_names=['A+','A-','B+','B-','O+','O-','AB+','AB-'],
                output_dict=True, zero_division=0
            )
        }
        self.results[model_name] = results
        return results

    def print_summary(self):
        print(f"\n{'='*70}")
        print("CROSS-VALIDATION SUMMARY")
        print(f"{'='*70}")
        for name, res in self.results.items():
            print(f"\n{name}:")
            print(f"  Accuracy: {res['accuracy_mean']:.4f} ± {res['accuracy_std']:.4f}")
            print(f"  95% CI (t): [{res['accuracy_95ci'][0]:.4f}, {res['accuracy_95ci'][1]:.4f}]")
            print(f"  Wilson 95% CI: [{res['wilson_95ci'][0]:.4f}, {res['wilson_95ci'][1]:.4f}]")
            print(f"  Macro F1: {res['f1_mean']:.4f} ± {res['f1_std']:.4f}")

# =============================================================================
# INFERENCE PROFILING
# =============================================================================

class InferenceProfiler:
    def __init__(self, config: Config):
        self.config = config
        self.device = config.device

    def profile(self, model: nn.Module, model_name: str, num_runs=100, batch_size=1):
        model.eval().to(self.device)
        dummy_img = torch.randn(batch_size, 3, self.config.img_size, self.config.img_size).to(self.device)
        dummy_mask = torch.randn(batch_size, 1, self.config.img_size, self.config.img_size).to(self.device)

        for _ in range(10):
            with torch.no_grad():
                _ = model(dummy_img, dummy_mask)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        times = []
        for _ in range(num_runs):
            start = time.perf_counter()
            with torch.no_grad():
                _ = model(dummy_img, dummy_mask)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            times.append(time.perf_counter() - start)

        memory_mb = 0
        if torch.cuda.is_available():
            memory_mb = torch.cuda.max_memory_allocated() / 1024**2
            torch.cuda.reset_peak_memory_stats()
        elif HAS_PSUTIL:
            memory_mb = psutil.Process(os.getpid()).memory_info().rss / 1024**2

        total_params = sum(p.numel() for p in model.parameters())
        model_size_mb = total_params * 4 / 1024**2

        results = {
            'model_name': model_name,
            'mean_latency_ms': np.mean(times) * 1000,
            'std_latency_ms': np.std(times) * 1000,
            'p95_latency_ms': np.percentile(times, 95) * 1000,
            'throughput_fps': batch_size / np.mean(times),
            'memory_mb': memory_mb,
            'model_size_mb': model_size_mb,
            'total_params': total_params,
            'device': self.device
        }
        return results

    def print_profile(self, r: Dict):
        print(f"\n{'='*60}")
        print(f"PROFILE: {r['model_name']}")
        print(f"{'='*60}")
        print(f"  Latency: {r['mean_latency_ms']:.2f} ± {r['std_latency_ms']:.2f} ms")
        print(f"  P95: {r['p95_latency_ms']:.2f} ms | Throughput: {r['throughput_fps']:.2f} FPS")
        print(f"  Params: {r['total_params']:,} | Size: {r['model_size_mb']:.2f} MB")
        print(f"  Memory: {r['memory_mb']:.2f} MB")
        feasible_pi = r['model_size_mb'] < 50 and r['mean_latency_ms'] < 50
        feasible_edge = r['model_size_mb'] < 20 and r['mean_latency_ms'] < 20
        print(f"  RPi4 Feasible: {'YES' if feasible_pi else 'NO (consider quantization)'}")
        print(f"  Edge TPU Feasible: {'YES' if feasible_edge else 'NEEDS QUANTIZATION'}")

# =============================================================================
# SENSITIVITY ANALYSIS
# =============================================================================

class SensitivityAnalyzer:
    def __init__(self, config: Config):
        self.config = config

    def _eval_loader(self, model, dataset):
        model.eval()
        loader = DataLoader(dataset, batch_size=self.config.batch_size, shuffle=False)
        correct, total = 0, 0
        with torch.no_grad():
            for images, masks, labels in loader:
                images = images.to(self.config.device)
                masks = masks.to(self.config.device)
                labels = labels.to(self.config.device)
                outputs = model(images, masks)
                _, predicted = outputs.max(1)
                correct += predicted.eq(labels).sum().item()
                total += labels.size(0)
        return correct / total if total > 0 else 0.0

    def analyze(self, model: nn.Module, base_dataset: BloodGroupDataset) -> Dict:
        print(f"\n{'='*70}")
        print("SENSITIVITY ANALYSIS")
        print(f"{'='*70}")
        results = {}
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        print("\n1. Mask Error Rate Sensitivity:")
        mask_res = {}
        for err in self.config.test_mask_error_rates:
            ds = BloodGroupDataset(base_dataset.split_dir, self.config,
                                    transform=transform, mode='test',
                                    mask_error_rate=err)
            ds.samples = base_dataset.samples
            ds.format_type = base_dataset.format_type
            acc = self._eval_loader(model, ds)
            mask_res[f"error_{err}"] = acc
            print(f"   Error {err:.1f}: Acc = {acc:.4f}")
        results['mask_errors'] = mask_res

        print("\n2. Lighting Variation Sensitivity:")
        light_res = {}
        for var in self.config.test_lighting_variations:
            ds = BloodGroupDataset(base_dataset.split_dir, self.config,
                                    transform=transform, mode='test',
                                    lighting_variant=var)
            ds.samples = base_dataset.samples
            ds.format_type = base_dataset.format_type
            acc = self._eval_loader(model, ds)
            light_res[var] = acc
            print(f"   {var:12s}: Acc = {acc:.4f}")
        results['lighting'] = light_res

        print("\n3. Weak Agglutination Simulation:")
        weak_res = {}
        for strength in [1.0, 0.8, 0.6, 0.4]:
            ds = BloodGroupDataset(base_dataset.split_dir, self.config,
                                    transform=transform, mode='test')
            ds.samples = base_dataset.samples
            ds.format_type = base_dataset.format_type
            loader = DataLoader(ds, batch_size=self.config.batch_size, shuffle=False)
            correct, total = 0, 0
            with torch.no_grad():
                for images, masks, labels in loader:
                    images = images.to(self.config.device)
                    noise = torch.randn_like(images) * (1 - strength) * 0.15
                    images = torch.clamp(images + noise, -2.5, 2.5)
                    masks = masks.to(self.config.device)
                    labels = labels.to(self.config.device)
                    outputs = model(images, masks)
                    _, predicted = outputs.max(1)
                    correct += predicted.eq(labels).sum().item()
                    total += labels.size(0)
            acc = correct / total if total > 0 else 0.0
            weak_res[f"strength_{strength}"] = acc
            print(f"   Strength {strength:.1f}: Acc = {acc:.4f}")
        results['weak_agglutination'] = weak_res

        return results

# =============================================================================
# VISUALIZATION
# =============================================================================

def plot_confusion_matrix(cm: np.ndarray, class_names: List[str], save_path: str, title: str = "Confusion Matrix"):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved confusion matrix: {save_path}")

def plot_training_history(history: Dict, save_path: str):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history['train_loss'], label='Train Loss')
    axes[0].plot(history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss Curves')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history['train_acc'], label='Train Acc')
    axes[1].plot(history['val_acc'], label='Val Acc')
    axes[1].set_title('Accuracy Curves')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved training curves: {save_path}")

# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    set_seed(42)
    config = Config()

    print("="*80)
    print("HemMaskNet - Reviewer 2 Response Pipeline (BUGFIXED)")
    print("="*80)
    print(f"Base Dir: {config.base_dir}")
    print(f"Device: {config.device}")
    print(f"Classes: {config.num_classes}")
    print(f"CV Folds: {config.n_splits}")
    print(f"Annotation Format: {config.annotation_format}")

    if not os.path.exists(config.base_dir):
        print(f"\nERROR: Base directory does not exist: {config.base_dir}")
        return

    print(f"\nBase directory contents: {os.listdir(config.base_dir)}")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Load datasets
    print("\n[1] Loading datasets...")
    train_dir = os.path.join(config.base_dir, 'train')
    valid_dir = os.path.join(config.base_dir, 'valid')
    test_dir = os.path.join(config.base_dir, 'test')

    train_ds = BloodGroupDataset(train_dir, config, transform=transform, mode='train')
    val_ds = BloodGroupDataset(valid_dir, config, transform=transform, mode='val')
    test_ds = BloodGroupDataset(test_dir, config, transform=transform, mode='test')

    if len(train_ds) == 0:
        print("\nERROR: No training samples found.")
        return

    # CRITICAL CHECK: if all labels are identical, something is wrong with format
    train_dist = train_ds.get_class_distribution()
    if len(train_dist) == 1:
        print("\n" + "!"*70)
        print("CRITICAL WARNING: All training samples have the same label.")
        print("This usually means the annotation format was misdetected.")
        print("If your dataset uses image-level blood group labels (Roboflow style),")
        print("set: config.annotation_format = 'image_level' before running.")
        print("!"*70 + "\n")
        # Don't exit; let user decide, but warn strongly

    # DataLoaders
    train_labels = [s['label'] for s in train_ds.samples]
    class_counts = np.bincount(train_labels, minlength=config.num_classes)
    weights = 1.0 / (class_counts + 1e-6)
    sample_weights = [weights[l] for l in train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    # FIX: drop_last=True on train_loader prevents BatchNorm crash when last batch has 1 sample
    train_loader = DataLoader(train_ds, batch_size=config.batch_size,
                              sampler=sampler, num_workers=0, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=config.batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=config.batch_size, shuffle=False, num_workers=0)

    # ================================================================
    # [2] TRAIN & EVALUATE ALL MODELS
    # ================================================================
    print("\n[2] Training and evaluating models...")

    models_to_test = [
        (HemMaskNet, "HemMaskNet"),
        (EfficientNetOnly, "EfficientNet-Only"),
        (ResNet50Baseline, "ResNet-50"),
        (MobileNetBaseline, "MobileNet-V2"),
        (ViTB16Baseline, "ViT-B/16"),
    ]

    all_test_results = {}
    all_histories = {}

    for model_class, name in models_to_test:
        print(f"\n--- Training {name} ---")
        model = model_class(config, num_classes=config.num_classes)
        trainer = Trainer(config, model, name)
        trainer.fit(train_loader, val_loader)

        test_loss, test_acc, preds, labels = trainer.evaluate(test_loader)
        all_test_results[name] = {
            'test_loss': test_loss,
            'test_acc': test_acc,
            'preds': preds,
            'labels': labels
        }
        all_histories[name] = trainer.history

        print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

        cm = confusion_matrix(labels, preds)
        plot_confusion_matrix(cm, train_ds.class_names,
                              os.path.join(config.output_dir, f'cm_{name.replace("/","_")}.png'),
                              title=f'{name} - Test Set Confusion Matrix')

        plot_training_history(trainer.history,
                              os.path.join(config.output_dir, f'history_{name.replace("/","_")}.png'))

    # ================================================================
    # [3] CROSS-VALIDATION
    # ================================================================
    print("\n[3] Running Stratified K-Fold Cross-Validation...")
    cv = CrossValidator(config)

    combined_samples = train_ds.samples + val_ds.samples
    combined_ds = BloodGroupDataset(train_dir, config, transform=transform, mode='train')
    combined_ds.samples = combined_samples
    combined_ds.format_type = train_ds.format_type

    for model_class, name in models_to_test:
        cv.run_cv(combined_ds, model_class, name)

    cv.print_summary()

    # ================================================================
    # [4] CLASSICAL BASELINE
    # ================================================================
    print("\n[4] Training Classical HOG-SVM Baseline...")
    classical = ClassicalBaseline()
    classical.fit(train_ds)
    y_true, y_pred = classical.predict(test_ds)
    classical_acc = accuracy_score(y_true, y_pred)
    classical_f1 = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)[2]
    print(f"  HOG-SVM Test Acc: {classical_acc:.4f}, Macro F1: {classical_f1:.4f}")
    all_test_results['HOG-SVM'] = {'test_acc': classical_acc}

    cm_classical = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm_classical, train_ds.class_names,
                          os.path.join(config.output_dir, 'cm_HOG_SVM.png'),
                          title='HOG-SVM - Test Set Confusion Matrix')

    # ================================================================
    # [5] INFERENCE PROFILING
    # ================================================================
    print("\n[5] Profiling Inference Speed & Memory...")
    profiler = InferenceProfiler(config)
    profile_results = []

    for model_class, name in models_to_test:
        model = model_class(config, num_classes=config.num_classes)
        prof = profiler.profile(model, name, num_runs=100)
        profiler.print_profile(prof)
        profile_results.append(prof)

    # ================================================================
    # [6] SENSITIVITY ANALYSIS
    # ================================================================
    print("\n[6] Running Sensitivity Analysis on HemMaskNet...")
    hem_mask = HemMaskNet(config, num_classes=config.num_classes)
    sens = SensitivityAnalyzer(config)
    sens_results = sens.analyze(hem_mask, test_ds)

    # ================================================================
    # [7] SAVE ALL RESULTS
    # ================================================================
    print("\n[7] Saving results...")

    with open(os.path.join(config.output_dir, 'cv_results.json'), 'w') as f:
        json.dump(cv.results, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else x)

    test_summary = {}
    for name, res in all_test_results.items():
        test_summary[name] = {
            'test_accuracy': float(res['test_acc']),
            'test_loss': float(res.get('test_loss', 0))
        }
    with open(os.path.join(config.output_dir, 'test_results.json'), 'w') as f:
        json.dump(test_summary, f, indent=2)

    with open(os.path.join(config.output_dir, 'inference_profiles.json'), 'w') as f:
        json.dump(profile_results, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else x)

    with open(os.path.join(config.output_dir, 'sensitivity_results.json'), 'w') as f:
        json.dump(sens_results, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.floating, np.integer)) else x)

    # ================================================================
    # [8] FINAL SUMMARY TABLE
    # ================================================================
    print("\n" + "="*80)
    print("FINAL COMPARISON TABLE (Reviewer 2)")
    print("="*80)
    print(f"{'Model':<20} {'Test Acc':<12} {'CV Acc':<12} {'CV F1':<12} {'Params (M)':<12} {'Latency (ms)':<14}")
    print("-"*80)

    for model_class, name in models_to_test:
        test_acc = all_test_results[name]['test_acc'] * 100
        cv_acc = cv.results[name]['accuracy_mean'] * 100 if name in cv.results else 0
        cv_f1 = cv.results[name]['f1_mean'] * 100 if name in cv.results else 0
        prof = next((p for p in profile_results if p['model_name'] == name), {})
        params = prof.get('total_params', 0) / 1e6
        lat = prof.get('mean_latency_ms', 0)
        print(f"{name:<20} {test_acc:<12.2f} {cv_acc:<12.2f} {cv_f1:<12.2f} {params:<12.2f} {lat:<14.2f}")

    print(f"{'HOG-SVM':<20} {classical_acc*100:<12.2f} {'N/A':<12} {classical_f1*100:<12.2f} {'N/A':<12} {'N/A':<14}")
    print("="*80)

    print(f"\nAll results saved to: {config.output_dir}")
    print("Done!")

if __name__ == "__main__":
    main()


HemMaskNet - Reviewer 2 Response Pipeline (BUGFIXED)
Base Dir: /home/fawadsalamkhan/MyProjects/BloodGroup
Device: cuda
Classes: 8
CV Folds: 5
Annotation Format: auto

Base directory contents: ['valid', 'test', 'train', 'data.yaml', 'results_reviewer2', 'README.roboflow.txt']

[1] Loading datasets...
  Found 1860 image files in /home/fawadsalamkhan/MyProjects/BloodGroup/train/images
  Auto-detected annotation format: drop_reactions
[train] Loaded 1857 samples from /home/fawadsalamkhan/MyProjects/BloodGroup/train
  Image dir: /home/fawadsalamkhan/MyProjects/BloodGroup/train/images
  Label dir: /home/fawadsalamkhan/MyProjects/BloodGroup/train/labels
  Class distribution: {1: 1857}
  Try setting config.annotation_format='image_level' if using Roboflow-style labels.
  Found 178 image files in /home/fawadsalamkhan/MyProjects/BloodGroup/valid/images
  Auto-detected annotation format: drop_reactions
[val] Loaded 178 samples from /home/fawadsalamkhan/MyProjects/BloodGroup/valid
  Image dir: /ho

ValueError: Number of classes, 1, does not match size of target_names, 8. Try specifying the labels parameter